## LIBRERIAS PARA QUE EL CODIGO FUNCIONE CORRECTAMENTE

In [2]:
!pip install psycopg2-binary
!pip install pandas

In [8]:
import pandas as pd
import psycopg2
# Conexión a la base de datos
conn = psycopg2.connect(
    dbname="postgres", 
    user="postgres",                 
    password="postgres",            
    host="localhost",                   
    port="5432"                    
)

# Cursor para interactuar con la base de datos
cursor = conn.cursor()

# 1. Consultar los Top 10 Productos más vendidos
consulta_productos = """
SELECT p.nombre_producto,
       SUM(hv.cantidad) AS total_vendido
FROM hecho_ventas hv
JOIN dim_producto p ON hv.id_producto = p.id_producto
GROUP BY p.id_producto, p.nombre_producto
ORDER BY total_vendido DESC
LIMIT 10;
"""
df_top_10_productos = pd.read_sql(consulta_productos, conn)
print(df_top_10_productos)

                  nombre_producto  total_vendido
0  AURICULARES INALAMBRICOS NTUNE            4.0
1       PANTALLA TACTIL 7"   AUTO            3.0
2     CARGADOR USB RAPIDO   COCHE            3.0
3     SOPORTE MAGNETICO   CELULAR            2.0
4               LAPTOP HP ENVY 13            2.0
5       RADIO DE AUTO PIONEER DEH            2.0
6         CAMARA DE REVERSA NTECH            2.0
7     LUCES LED INTERIORES   AUTO            2.0
8               ALTAVOZ SONY XB33            2.0
9     SMARTWATCH XIAOMI MI BAND 7            2.0
                   cliente  numero_pedidos
0  JOSE ANDRES MUNOZ PEREZ               4
1     FERNANDA PEREZ LOPEZ               4
2           CARLOS RAMIREZ               4
3  MARIA JULIA GOMEZ NUNEZ               4
4       ANA LUCIA ZAMBRANO               2


/tmp/ipykernel_25294/1465330273.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_10_productos = pd.read_sql(consulta_productos, conn)
/tmp/ipykernel_25294/1465330273.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_5_clientes = pd.read_sql(consulta_clientes, conn)


In [ ]:
# 2. Consultar los Top 5 Clientes con el mayor número de pedidos
consulta_clientes = """
SELECT c.nombres || ' ' || c.apellidos AS cliente,
       COUNT(hv.id_cliente) AS numero_pedidos
FROM hecho_ventas hv
JOIN dim_cliente c ON hv.id_cliente = c.id_cliente
GROUP BY c.id_cliente, c.nombres, c.apellidos
ORDER BY numero_pedidos DESC
LIMIT 5;
"""
df_top_5_clientes = pd.read_sql(consulta_clientes, conn)
print(df_top_5_clientes)

In [ ]:
# 3. Top 5 Corresponsales con el mayor número de pedidos
consulta_corresponsales = """
SELECT cor.nombre_corresponsal,
       COUNT(hv.id_corresponsal) AS numero_pedidos
FROM hecho_ventas hv
JOIN dim_corresponsal cor ON hv.id_corresponsal = cor.id_corresponsal
GROUP BY cor.id_corresponsal, cor.nombre_corresponsal
ORDER BY numero_pedidos DESC
LIMIT 5;
"""
df_top_5_corresponsales = pd.read_sql(consulta_corresponsales, conn)
df_top_5_corresponsales.head()

In [12]:
# 4. Total de pagos diario y mensual por productos
consulta_pagos_diarios_mensuales_productos = """
SELECT p.nombre_producto,
       TO_CHAR(CAST(t.fecha AS DATE), 'YYYY-MM-DD') AS fecha_diaria,
       SUM(hv.total_pagado) AS total_pagado_diario,
       TO_CHAR(CAST(t.fecha AS DATE), 'YYYY-MM') AS fecha_mensual,
       -- Subconsulta para calcular el total mensual por producto
       (SELECT SUM(hv2.total_pagado)
        FROM hecho_ventas hv2
        JOIN dim_tiempo t2 ON hv2.id_fecha = t2.id_fecha
        WHERE p.id_producto = hv2.id_producto
        AND TO_CHAR(CAST(t2.fecha AS DATE), 'YYYY-MM') = TO_CHAR(CAST(t.fecha AS DATE), 'YYYY-MM')
       ) AS total_pagado_mensual
FROM hecho_ventas hv
JOIN dim_producto p ON hv.id_producto = p.id_producto
JOIN dim_tiempo t ON hv.id_fecha = t.id_fecha
GROUP BY p.id_producto, p.nombre_producto, t.fecha
ORDER BY t.fecha;
"""
df_pagos_productos = pd.read_sql(consulta_pagos_diarios_mensuales_productos, conn)
df_pagos_productos.head()
conn.close()

/tmp/ipykernel_25294/820893318.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_5_corresponsales = pd.read_sql(consulta_corresponsales, conn)
/tmp/ipykernel_25294/820893318.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pagos_productos = pd.read_sql(consulta_pagos_diarios_mensuales_productos, conn)


,nombre_producto,fecha_diaria,total_pagado_diario,fecha_mensual,total_pagado_mensual
0,"TELEVISOR LED 50"" 4K",2025-04-29,550.00,2025-04,1100.00
1,LAPTOP HP ENVY 13,2025-04-29,1799.02,2025-04,1799.02
2,AURICULARES INALAMBRICOS NTUNE,2025-04-29,130.00,2025-04,520.00
3,ALTAVOZ SONY XB33,2025-04-29,298.26,2025-04,298.26
4,RADIO DE AUTO PIONEER DEH,2025-04-29,90.00,2025-04,180.00
